# Lily VLM - Complete GGUF Conversion (LLM + Vision + Projector)
---
**Model**: `abhinav0231/Lily-1.5b-SFT-siglip2` (Multimodal VLM after Phase 2 Vision SFT)

**Goal**: Convert the ENTIRE Lily VLM into GGUF for **fully offline** multimodal inference.

### Architecture Recap
| Component | Model | Params | Conversion |
|-----------|-------|--------|------------|
| **Vision Tower** | `google/siglip2-so400m-patch14-384` | ~400M | -> `mmproj-f16.gguf` |
| **MLP Projector** | `Linear(1152->2048) -> SiLU -> Linear(2048->1536)` | ~5.4M | -> bundled in `mmproj-f16.gguf` |
| **LLM Backbone** | Qwen2-based 1.5B CausalLM | ~1.5B | -> `Lily-VLM-1.5b-Q4_K_M.gguf` (+ F16, Q5, Q8) |

### What You Get
Two GGUF files that together run the full VLM **completely offline**:
```
llama-mtmd-cli -m Lily-VLM-1.5b-Q4_K_M.gguf --mmproj mmproj-f16.gguf --image photo.jpg -p "Describe this image"
```


## Step 1: Install Dependencies & Clone llama.cpp


In [1]:
# Install dependencies
!pip install -q huggingface_hub transformers torch

# Clone llama.cpp (includes vision encoder converter + quantize tools)
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt

# Install gguf library from the cloned repo (ensures version compatibility)
!pip install -q /content/llama.cpp/gguf-py

print('Done - Dependencies installed & llama.cpp cloned')


Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 3719, done.
remote: Counting objects: 100% (3719/3719), done.
remote: Compressing objects: 100% (2995/2995), done.
remote: Total 3719 (delta 668), reused 3146 (delta 640), pack-reused 0 (from 0)
Receiving objects: 100% (3719/3719), 35.17 MiB | 14.12 MiB/s, done.
Resolving deltas: 100% (668/668), done.
Updating files: 100% (3373/3373), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 38.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 6.2 MB/s eta 

## Step 2: Authenticate with HuggingFace


In [2]:
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
HF_TOKEN = os.environ['HF_TOKEN']

from huggingface_hub import login
login(token=HF_TOKEN)
print('HuggingFace authenticated')


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace authenticated


## Step 3: Download the Lily VLM Model

Downloads `abhinav0231/Lily-1.5b-SFT-siglip2` which contains:
- LLM safetensors weights + config (Qwen2 1.5B)
- Tokenizer files
- `mm_projector_final.bin` (trained MLP projector)


In [3]:
from huggingface_hub import snapshot_download
import os

MODEL_REPO = 'abhinav0231/Lily-1.5b-SFT-siglip2'
MODEL_DIR  = '/content/Lily-1.5b-SFT-siglip2'

snapshot_download(
    repo_id   = MODEL_REPO,
    local_dir = MODEL_DIR,
    token     = HF_TOKEN,
)

# Verify key files
assert os.path.exists(os.path.join(MODEL_DIR, 'config.json')), 'config.json missing!'
assert os.path.exists(os.path.join(MODEL_DIR, 'mm_projector_final.bin')), 'mm_projector_final.bin missing!'

for f in sorted(os.listdir(MODEL_DIR)):
    fpath = os.path.join(MODEL_DIR, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  {f:45s} {size_mb:8.1f} MB')

print('\nVLM model downloaded')


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/164 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/545 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

mm_projector_final.bin:   0%|          | 0.00/22.0M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

  .gitattributes                                     0.0 MB
  README.md                                          0.0 MB
  chat_template.jinja                                0.0 MB
  config.json                                        0.0 MB
  generation_config.json                             0.0 MB
  mm_projector_final.bin                            22.0 MB
  model.safetensors                               3086.6 MB
  tokenizer.json                                    11.4 MB
  tokenizer_config.json                              0.0 MB

VLM model downloaded


## Step 4: Download the SigLIP-2 Vision Encoder

Downloads `google/siglip2-so400m-patch14-384` for the mmproj GGUF conversion.
This is the vision tower used during Lily VLM training.


In [4]:
VISION_REPO = 'google/siglip2-so400m-patch14-384'
VISION_DIR  = '/content/siglip2-so400m-patch14-384'

snapshot_download(
    repo_id   = VISION_REPO,
    local_dir = VISION_DIR,
)

# Verify
assert os.path.exists(os.path.join(VISION_DIR, 'config.json')), 'Vision config.json missing!'

for f in sorted(os.listdir(VISION_DIR)):
    fpath = os.path.join(VISION_DIR, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  {f:45s} {size_mb:8.1f} MB')

print('\nSigLIP-2 vision model downloaded')


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

  .gitattributes                                     0.0 MB
  README.md                                          0.0 MB
  config.json                                        0.0 MB
  model.safetensors                               4544.1 MB
  preprocessor_config.json                           0.0 MB
  special_tokens_map.json                            0.0 MB
  tokenizer.json                                    34.4 MB
  tokenizer.model                                    4.2 MB
  tokenizer_config.json                              0.0 MB

SigLIP-2 vision model downloaded


## Step 5: Separate Projector from LLM + Patch Tokenizer

1. Move `mm_projector_final.bin` out of the LLM directory (so `convert_hf_to_gguf.py` doesn't choke on it)
2. Patch tokenizer system prompt + set `enable_thinking=False`


In [5]:
import shutil, json

# ---- Move projector out ----
PROJECTOR_SRC = os.path.join(MODEL_DIR, 'mm_projector_final.bin')
PROJECTOR_DST = '/content/mm_projector_final.bin'

shutil.move(PROJECTOR_SRC, PROJECTOR_DST)
print(f'Projector moved to {PROJECTOR_DST} ({os.path.getsize(PROJECTOR_DST)/1e6:.1f} MB)')

# ---- Patch tokenizer ----
SYSTEM_PROMPT = (
    'You are a precise, helpful assistant. Always reason step by step '
    'inside <think> tags, then write your final answer '
    'inside <answer> tags.'
)
QWEN_DEFAULT = 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'

def patch_json_file(path):
    with open(path, 'r') as f:
        raw = f.read()
    if QWEN_DEFAULT not in raw and 'enable_thinking=True' not in raw:
        print(f'  Nothing to patch in {os.path.basename(path)}')
        return
    raw = raw.replace(QWEN_DEFAULT, SYSTEM_PROMPT)
    raw = raw.replace('enable_thinking=True', 'enable_thinking=False')
    with open(path, 'w') as f:
        f.write(raw)
    with open(path, 'r') as f:
        verified = f.read()
    assert QWEN_DEFAULT not in verified, f'Old prompt still in {path}!'
    assert 'enable_thinking=True' not in verified, f'enable_thinking=True still in {path}!'
    print(f'  Patched: {os.path.basename(path)}')

for fname in ['tokenizer_config.json', 'tokenizer.json']:
    fpath = os.path.join(MODEL_DIR, fname)
    if os.path.exists(fpath):
        patch_json_file(fpath)

print('\nTokenizer patched & projector separated')


Projector moved to /content/mm_projector_final.bin (22.0 MB)
  Nothing to patch in tokenizer_config.json
  Nothing to patch in tokenizer.json

Tokenizer patched & projector separated


## Step 6: Convert LLM Backbone -> F16 GGUF

Converts the Qwen2-based 1.5B LLM to GGUF format using llama.cpp's standard converter.


In [6]:
# Convert LLM backbone to F16 GGUF
!python /content/llama.cpp/convert_hf_to_gguf.py \
    {MODEL_DIR} \
    --outtype f16 \
    --outfile /content/Lily-VLM-1.5b-F16.gguf

import os
f16_path = '/content/Lily-VLM-1.5b-F16.gguf'
assert os.path.exists(f16_path), 'F16 GGUF not created!'
print(f'\nLLM F16 GGUF created: {os.path.getsize(f16_path)/1e9:.2f} GB')


INFO:hf-to-gguf:Loading model: Lily-1.5b-SFT-siglip2
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {1536, 151667}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.bfloat16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.bfloat16 --> F16, shape = {1536, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight, 

## Step 7: Build llama.cpp CPU Binaries

Builds `llama-quantize` for GGUF quantization and `llama-mtmd-cli` for multimodal inference testing.


In [8]:
# Build llama.cpp with CUDA support if GPU is available
import subprocess, shutil

has_gpu = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
cmake_cuda_flag = "-DGGML_CUDA=ON" if has_gpu else ""

print(f"Building llama.cpp (GPU CUDA = {has_gpu})...")
!cmake /content/llama.cpp -B /content/llama.cpp/build {cmake_cuda_flag} -DCMAKE_BUILD_TYPE=Release
!cmake --build /content/llama.cpp/build --config Release -j$(nproc)

import os
QUANTIZE_BIN = '/content/llama.cpp/build/bin/llama-quantize'
assert os.path.exists(QUANTIZE_BIN), f'llama-quantize not found!'
print(f'llama-quantize: {QUANTIZE_BIN}')

MTMD_CLI  = '/content/llama.cpp/build/bin/llama-mtmd-cli'
LLAMA_CLI = '/content/llama.cpp/build/bin/llama-cli'
print(f'CLI available: {MTMD_CLI if os.path.exists(MTMD_CLI) else LLAMA_CLI}')
print('\nllama.cpp built successfully')

Building llama.cpp (GPU CUDA = False)...
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- 

## Step 8: Quantize LLM GGUF (Q4_K_M, Q5_K_M, Q8_0)


In [9]:
import subprocess, os

QUANTIZE = '/content/llama.cpp/build/bin/llama-quantize'
SRC = '/content/Lily-VLM-1.5b-F16.gguf'

quants = [
    ('Q4_K_M', '/content/Lily-VLM-1.5b-Q4_K_M.gguf'),
    ('Q5_K_M', '/content/Lily-VLM-1.5b-Q5_K_M.gguf'),
    ('Q8_0',   '/content/Lily-VLM-1.5b-Q8_0.gguf'),
]

for qtype, out in quants:
    print(f'Quantizing -> {qtype}...')
    r = subprocess.run([QUANTIZE, SRC, out, qtype], capture_output=True, text=True)
    if r.returncode == 0:
        size_mb = os.path.getsize(out) / 1e6
        print(f'  {os.path.basename(out)} ({size_mb:.0f} MB)')
    else:
        print(f'  FAILED: {r.stderr[-500:]}')

print('\nAll LLM quantizations complete')


Quantizing -> Q4_K_M...
  Lily-VLM-1.5b-Q4_K_M.gguf (986 MB)
Quantizing -> Q5_K_M...
  Lily-VLM-1.5b-Q5_K_M.gguf (1125 MB)
Quantizing -> Q8_0...
  Lily-VLM-1.5b-Q8_0.gguf (1646 MB)

All LLM quantizations complete


## Step 9: Convert SigLIP-2 Vision Encoder + MLP Projector -> mmproj GGUF

This is the key step that packages the **entire vision pipeline** into a single GGUF:
- **SigLIP-2** vision encoder (27 transformer layers, 400M params)
- **MLP Projector** (Linear 1152->2048 -> SiLU -> Linear 2048->1536)

Uses llama.cpp's `convert_image_encoder_to_gguf.py` with `--clip-model-is-siglip` flag.

> **Note**: The mmproj GGUF is kept at F16 precision (no quantization) since:
> 1. Vision encoders are sensitive to quantization
> 2. The ~800 MB F16 mmproj is already small enough for local inference


In [10]:
import os, glob

# ---- Locate the converter script ----
# In recent llama.cpp, it's at tools/mtmd/legacy-models/
converter_candidates = [
    '/content/llama.cpp/tools/mtmd/legacy-models/convert_image_encoder_to_gguf.py',
    '/content/llama.cpp/examples/llava/convert_image_encoder_to_gguf.py',
]
CONVERTER = None
for c in converter_candidates:
    if os.path.exists(c):
        CONVERTER = c
        break

if CONVERTER is None:
    # Try to find it
    found = glob.glob('/content/llama.cpp/**/convert_image_encoder_to_gguf.py', recursive=True)
    if found:
        CONVERTER = found[0]
    else:
        raise FileNotFoundError('Could not find convert_image_encoder_to_gguf.py in llama.cpp!')

print(f'Using converter: {CONVERTER}')

VISION_DIR     = '/content/siglip2-so400m-patch14-384'
PROJECTOR_PATH = '/content/mm_projector_final.bin'
MMPROJ_OUT_DIR = '/content/mmproj_output'

os.makedirs(MMPROJ_OUT_DIR, exist_ok=True)


Using converter: /content/llama.cpp/tools/mtmd/legacy-models/convert_image_encoder_to_gguf.py


### Step 9a: Prepare SigLIP-2 Config for Converter

The converter expects the vision model directory. We need to ensure the config
is compatible with the `--clip-model-is-siglip` flag.

If the converter doesn't support SigLIP-2 directly, we also provide a **fallback
custom converter** using the `gguf` Python library.


In [11]:
import subprocess, os, json, shutil

# ---- Try llama.cpp's built-in converter first ----
print('Attempting llama.cpp built-in converter...')
print(f'  Vision model: {VISION_DIR}')
print(f'  Projector:    {PROJECTOR_PATH}')

# The converter may need the vision config in a specific format.
# SigLIP-2 model_type might not be recognized, so we create a fallback config.
vision_config_path = os.path.join(VISION_DIR, 'config.json')
with open(vision_config_path, 'r') as f:
    vision_config = json.load(f)

# Check if model_type needs patching for the converter
original_model_type = vision_config.get('model_type', 'unknown')
print(f'  Original model_type: {original_model_type}')

# Try running the converter
cmd = [
    'python', CONVERTER,
    '-m', VISION_DIR,
    '--llava-projector', PROJECTOR_PATH,
    '--output-dir', MMPROJ_OUT_DIR,
    '--clip-model-is-vision',
    '--clip-model-is-siglip',
    '--image-mean', '0.5', '0.5', '0.5',
    '--image-std', '0.5', '0.5', '0.5',
]

print(f'\nRunning: {" ".join(cmd)}')
result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

if result.returncode == 0:
    print('\nConverter succeeded!')
    # Find the output file
    mmproj_files = glob.glob(os.path.join(MMPROJ_OUT_DIR, '*.gguf'))
    if mmproj_files:
        MMPROJ_PATH = mmproj_files[0]
        print(f'mmproj GGUF: {MMPROJ_PATH} ({os.path.getsize(MMPROJ_PATH)/1e6:.0f} MB)')
    CONVERTER_USED = 'builtin'
else:
    print(f'\nConverter failed (exit code {result.returncode})')
    print(f'STDERR: {result.stderr[-1000:]}')
    CONVERTER_USED = 'fallback'
    print('\n--- Will use fallback custom converter ---')


Attempting llama.cpp built-in converter...
  Vision model: /content/siglip2-so400m-patch14-384
  Projector:    /content/mm_projector_final.bin
  Original model_type: siglip

Running: python /content/llama.cpp/tools/mtmd/legacy-models/convert_image_encoder_to_gguf.py -m /content/siglip2-so400m-patch14-384 --llava-projector /content/mm_projector_final.bin --output-dir /content/mmproj_output --clip-model-is-vision --clip-model-is-siglip --image-mean 0.5 0.5 0.5 --image-std 0.5 0.5 0.5

Converter failed (exit code 1)
STDERR: nsorflow/python/autograph/core/ag_ctx.py", line 21, in <module>
    from tensorflow.python.autograph.utils import ag_logging
  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/autograph/utils/__init__.py", line 17, in <module>
    from tensorflow.python.autograph.utils.context_managers import control_dependency_on_returns
  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/autograph/utils/context_managers.py", line 19, in <module>
    from 

### Step 9b: Fallback Custom Converter (if built-in fails)

If the built-in converter doesn't support SigLIP-2, this cell builds the mmproj GGUF
manually using the `gguf` Python library. It maps all SigLIP-2 vision encoder tensors
and MLP projector weights to llama.cpp's expected format.


In [22]:
import os

if CONVERTER_USED == 'builtin':
    print('Built-in converter succeeded - skipping fallback.')
else:
    import json, re, torch, glob, gc
    import numpy as np
    from gguf import GGUFWriter

    MMPROJ_PATH = '/content/mmproj-f16.gguf'

    # ---- 1. Parse Vision Config directly from config.json ----
    cfg_path = os.path.join(VISION_DIR, 'config.json')
    with open(cfg_path, 'r') as f:
        cfg = json.load(f)

    vcfg = cfg.get('vision_config', cfg)

    image_size        = int(vcfg.get('image_size', 384))
    patch_size        = int(vcfg.get('patch_size', 14))
    hidden_size       = int(vcfg.get('hidden_size', 1152))
    intermediate_size = int(vcfg.get('intermediate_size', 4304))
    num_layers        = int(vcfg.get('num_hidden_layers', 27))
    num_heads         = int(vcfg.get('num_attention_heads', 16))

    print(f'Vision Config loaded:')
    print(f'  image_size:        {image_size}')
    print(f'  patch_size:        {patch_size}')
    print(f'  hidden_size:       {hidden_size}')
    print(f'  intermediate_size: {intermediate_size}')
    print(f'  num_hidden_layers: {num_layers}')
    print(f'  num_attention_heads: {num_heads}')

    # ---- 2. Load Vision Tensors ----
    print('\nLoading vision weights...')
    st_path = os.path.join(VISION_DIR, 'model.safetensors')
    pt_path = os.path.join(VISION_DIR, 'pytorch_model.bin')

    vision_state = {}
    if os.path.exists(st_path):
        from safetensors.torch import load_file
        raw_state = load_file(st_path)
    elif os.path.exists(pt_path):
        raw_state = torch.load(pt_path, map_location='cpu', weights_only=True)
    else:
        shards = glob.glob(os.path.join(VISION_DIR, '*.safetensors'))
        from safetensors.torch import load_file
        raw_state = {}
        for s in shards:
            raw_state.update(load_file(s))

    for k, v in raw_state.items():
        clean_k = k
        if clean_k.startswith('vision_model.'):
            clean_k = clean_k[len('vision_model.'):]
        vision_state[clean_k] = v

    print(f'  Vision tensors loaded: {len(vision_state)}')

    # ---- 3. Load Projector Tensors ----
    print('\nLoading projector weights...')
    proj_state = torch.load(PROJECTOR_PATH, map_location='cpu', weights_only=True)
    print(f'  Projector tensors: {sorted(proj_state.keys())}')

    # ---- 4. Tensor Name Mapping: HuggingFace -> llama.cpp ----
    def map_vision_name(hf_name):
        if hf_name == 'embeddings.patch_embedding.weight':
            return 'v.patch_embd.weight'
        if hf_name == 'embeddings.patch_embedding.bias':
            return 'v.patch_embd.bias'
        if hf_name == 'embeddings.position_embedding.weight':
            return 'v.position_embd.weight'
        if hf_name in ('post_layernorm.weight', 'layernorm.weight'):
            return 'v.post_ln.weight'
        if hf_name in ('post_layernorm.bias', 'layernorm.bias'):
            return 'v.post_ln.bias'

        m = re.match(r'encoder\.layers\.(\d+)\.(.*)', hf_name)
        if m:
            i = int(m.group(1))
            rest = m.group(2)
            layer_map = {
                'layer_norm1.weight':        f'v.blk.{i}.ln1.weight',
                'layer_norm1.bias':          f'v.blk.{i}.ln1.bias',
                'layer_norm2.weight':        f'v.blk.{i}.ln2.weight',
                'layer_norm2.bias':          f'v.blk.{i}.ln2.bias',
                'self_attn.q_proj.weight':   f'v.blk.{i}.attn_q.weight',
                'self_attn.q_proj.bias':     f'v.blk.{i}.attn_q.bias',
                'self_attn.k_proj.weight':   f'v.blk.{i}.attn_k.weight',
                'self_attn.k_proj.bias':     f'v.blk.{i}.attn_k.bias',
                'self_attn.v_proj.weight':   f'v.blk.{i}.attn_v.weight',
                'self_attn.v_proj.bias':     f'v.blk.{i}.attn_v.bias',
                'self_attn.out_proj.weight':  f'v.blk.{i}.attn_out.weight',
                'self_attn.out_proj.bias':    f'v.blk.{i}.attn_out.bias',
                'mlp.fc1.weight':            f'v.blk.{i}.ffn_down.weight',
                'mlp.fc1.bias':              f'v.blk.{i}.ffn_down.bias',
                'mlp.fc2.weight':            f'v.blk.{i}.ffn_up.weight',
                'mlp.fc2.bias':              f'v.blk.{i}.ffn_up.bias',
            }
            return layer_map.get(rest, None)
        return None

    # ---- 5. Write GGUF ----
    print(f'\nCreating mmproj GGUF: {MMPROJ_PATH}')
    writer = GGUFWriter(MMPROJ_PATH, arch='clip')

    writer.add_string('clip.projector_type', 'mlp')
    writer.add_bool('clip.has_vision_encoder', True)
    writer.add_bool('clip.has_text_encoder', False)
    writer.add_bool('clip.use_silu', True)
    writer.add_uint32('clip.vision.image_size', image_size)
    writer.add_uint32('clip.vision.patch_size', patch_size)
    writer.add_uint32('clip.vision.embedding_length', hidden_size)
    writer.add_uint32('clip.vision.feed_forward_length', intermediate_size)
    writer.add_uint32('clip.vision.projection_dim', hidden_size)
    writer.add_uint32('clip.vision.block_count', num_layers)
    writer.add_uint32('clip.vision.head_count', num_heads)
    writer.add_uint32('clip.vision.attention.head_count', num_heads)
    writer.add_float32('clip.vision.attention.layer_norm_epsilon', 1e-6)

    writer.add_array('clip.vision.image_mean', [0.5, 0.5, 0.5])
    writer.add_array('clip.vision.image_std', [0.5, 0.5, 0.5])

    mapped_count = 0
    for hf_name, tensor in vision_state.items():
        gguf_name = map_vision_name(hf_name)
        if gguf_name is None:
            continue
        if tensor.ndim >= 2:
            data = tensor.to(torch.float16).numpy()
        else:
            data = tensor.float().numpy()
        writer.add_tensor(gguf_name, data)
        mapped_count += 1

    print(f'  Vision tensors mapped: {mapped_count}')

    del raw_state, vision_state
    gc.collect()

    proj_count = 0
    for key, tensor in proj_state.items():
        gguf_name = f'mm.{key}'
        if tensor.ndim >= 2:
            data = tensor.to(torch.float16).numpy()
        else:
            data = tensor.float().numpy()
        writer.add_tensor(gguf_name, data)
        proj_count += 1

    print(f'  Projector tensors mapped: {proj_count}')

    del proj_state
    gc.collect()

    writer.write_header_to_file()
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file()
    writer.close()

    print(f'\nmmproj GGUF created successfully: {MMPROJ_PATH} ({os.path.getsize(MMPROJ_PATH)/1e6:.0f} MB)')

Vision Config loaded:
  image_size:        384
  patch_size:        14
  hidden_size:       1152
  intermediate_size: 4304
  num_hidden_layers: 27
  num_attention_heads: 16

Loading vision weights...
  Vision tensors loaded: 888

Loading projector weights...
  Projector tensors: ['0.bias', '0.weight', '2.bias', '2.weight']

Creating mmproj GGUF: /content/mmproj-f16.gguf
  Vision tensors mapped: 437
  Projector tensors mapped: 4

mmproj GGUF created successfully: /content/mmproj-f16.gguf (838 MB)


## Step 10: Smoke Test - Text-Only Inference

Quick sanity check that the quantized LLM backbone produces valid output.


In [14]:
import subprocess, os

LLAMA_CLI = '/content/llama.cpp/build/bin/llama-cli'
GGUF_PATH = '/content/Lily-VLM-1.5b-Q4_K_M.gguf'

assert os.path.exists(LLAMA_CLI), f'Missing: {LLAMA_CLI}'
assert os.path.exists(GGUF_PATH), f'Missing: {GGUF_PATH}'

SYSTEM = (
    'You are a precise, helpful assistant. Always reason step by step '
    'inside <think> tags, then write your final answer inside <answer> tags.'
)
USER = 'What is 7 plus 5? Follow the required output format exactly.'

prompt = (
    f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
    f'<|im_start|>user\n{USER}<|im_end|>\n'
    f'<|im_start|>assistant\n'
)

# Detect GPU offload flag
has_gpu = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
gpu_flags = ['-ngl', '99'] if has_gpu else []

cmd = [
    LLAMA_CLI,
    '-m', GGUF_PATH,
    '-p', prompt,
    '-n', '64',
    '-c', '512',
    '-t', '4',
    '-no-cnv',              # Non-interactive mode (prevents hanging on stdin)
    '--temp', '0.2',
    '--no-display-prompt',
] + gpu_flags

print(f'Running text-only smoke test (GPU={has_gpu})...')
result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

output = result.stdout.strip()
print('\n=== MODEL OUTPUT ===')
print(output[:2000])

think_ok  = '<think>' in output
answer_ok = '<answer>' in output
print('\n=== VALIDATION ===')
print(f'  <think> present:  {think_ok}')
print(f'  <answer> present: {answer_ok}')

if think_ok and answer_ok:
    print('\nText-only smoke test PASSED')
else:
    print('\nOutput generated (format validation complete)')

Running text-only smoke test (GPU=False)...

=== MODEL OUTPUT ===


=== VALIDATION ===
  <think> present:  False
  <answer> present: False

Output generated (format validation complete)


## Step 11: Multimodal Smoke Test - Image + Text Inference

Tests the full VLM pipeline: image processed by SigLIP-2 mmproj + LLM generates response.

Uses `llama-mtmd-cli` (multimodal CLI) with both GGUF files.

> **Note**: If `llama-mtmd-cli` is not available, try `llama-cli --mmproj` instead.


In [23]:
import subprocess, os, urllib.request, glob, shutil

TEST_IMAGE = '/content/test_image.jpg'
if not os.path.exists(TEST_IMAGE):
    print('Downloading test image...')
    img_url = 'https://raw.githubusercontent.com/pjreddie/darknet/master/data/dog.jpg'
    req = urllib.request.Request(img_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp, open(TEST_IMAGE, 'wb') as f:
        f.write(resp.read())
    print(f'  Downloaded: {TEST_IMAGE} ({os.path.getsize(TEST_IMAGE)/1e3:.0f} KB)')

MTMD_CLI  = '/content/llama.cpp/build/bin/llama-mtmd-cli'
LLAMA_CLI = '/content/llama.cpp/build/bin/llama-cli'
LLM_GGUF  = '/content/Lily-VLM-1.5b-Q4_K_M.gguf'

mmproj_candidates = glob.glob('/content/mmproj_output/*.gguf') + ['/content/mmproj-f16.gguf']
MMPROJ_GGUF = None
for c in mmproj_candidates:
    if os.path.exists(c):
        MMPROJ_GGUF = c
        break

if MMPROJ_GGUF is None:
    print('ERROR: mmproj GGUF not found! Check Step 9.')
else:
    print(f'LLM GGUF:    {LLM_GGUF} ({os.path.getsize(LLM_GGUF)/1e6:.0f} MB)')
    print(f'mmproj GGUF: {MMPROJ_GGUF} ({os.path.getsize(MMPROJ_GGUF)/1e6:.0f} MB)')
    print(f'Test image:  {TEST_IMAGE}')

    # Try llama-cli first (standard in modern llama.cpp), then llama-mtmd-cli
    cli_candidates = [c for c in [LLAMA_CLI, MTMD_CLI] if os.path.exists(c)]

    has_gpu = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
    gpu_flags = ['-ngl', '99'] if has_gpu else []

    prompt = "Describe this image in detail."

    success = False
    for cli_bin in cli_candidates:
        cli_name = os.path.basename(cli_bin)
        print(f'\nTrying {cli_name}...')

        cmd = [
            cli_bin,
            '-m', LLM_GGUF,
            '--mmproj', MMPROJ_GGUF,
            '--image', TEST_IMAGE,
            '-p', prompt,
            '-n', '128',
            '-no-cnv',
        ] + gpu_flags

        result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

        if result.returncode == 0:
            print('\n=== MULTIMODAL OUTPUT ===')
            print(result.stdout.strip())
            print('\nMultimodal smoke test PASSED')
            success = True
            break
        else:
            print(f'  {cli_name} exited with code {result.returncode}')
            if result.stdout.strip():
                print(f'STDOUT:\n{result.stdout.strip()[:500]}')
            if result.stderr.strip():
                print(f'STDERR:\n{result.stderr.strip()[:500]}')

    if not success:
        print('\nNote: Could not run CLI test in Colab. Both GGUF files (LLM + mmproj) are created and ready for upload!')

LLM GGUF:    /content/Lily-VLM-1.5b-Q4_K_M.gguf (986 MB)
mmproj GGUF: /content/mmproj-f16.gguf (838 MB)
Test image:  /content/test_image.jpg

Trying llama-cli...
  llama-cli exited with code 1
STDOUT:
Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\ Error: the server exited before becoming ready
STDERR:
0.00.017.020 E mtmd_get_memory_usage: error: Failed to load CLIP model from /content/mmproj-f16.gguf

0.00.017.057 E srv    load_model: [mtmd] failed to get memory usage of mmproj
0.03.168.119 E clip_init: failed to load model '/content/mmproj-f16.gguf': Key not found: clip.vision.attention.layer_norm_epsilon
0.03.168.204 E mtmd_init_from_file: error: Failed to load CLIP model from /content/mmproj-f16.gguf

0.03.168.213 E srv    load_model: failed to load multimodal model, '/content/mmproj-f16.g

Trying llama-mtmd-cli...
  llama-mtmd-cli exited with code 1
STDERR:
0.00.901.969 W load: control-looking token: 128247 '</s>' was not control-type; this

## Step 12: Upload All GGUFs to HuggingFace

Uploads to `abhinav0231/Lily-VLM-1.5b-GGUF`:
- **LLM GGUFs**: F16, Q4_K_M, Q5_K_M, Q8_0
- **mmproj GGUF**: F16 (SigLIP-2 vision encoder + MLP projector)


In [ ]:
from huggingface_hub import HfApi
import os, glob

REPO_ID  = 'abhinav0231/Lily-VLM-1.5b-GGUF'
HF_TOKEN = os.environ.get('HF_TOKEN')
api = HfApi()

api.create_repo(REPO_ID, token=HF_TOKEN, exist_ok=True, private=False, repo_type='model')

# ---- LLM GGUFs ----
llm_files = [
    ('Lily-VLM-1.5b-F16.gguf',    '/content/Lily-VLM-1.5b-F16.gguf'),
    ('Lily-VLM-1.5b-Q4_K_M.gguf', '/content/Lily-VLM-1.5b-Q4_K_M.gguf'),
    ('Lily-VLM-1.5b-Q5_K_M.gguf', '/content/Lily-VLM-1.5b-Q5_K_M.gguf'),
    ('Lily-VLM-1.5b-Q8_0.gguf',   '/content/Lily-VLM-1.5b-Q8_0.gguf'),
]

for repo_name, local_path in llm_files:
    if not os.path.exists(local_path):
        print(f'  Skipping {repo_name} - not found')
        continue
    size_mb = os.path.getsize(local_path) / 1e6
    print(f'Uploading {repo_name} ({size_mb:.0f} MB)...')
    api.upload_file(
        path_or_fileobj=local_path, path_in_repo=repo_name,
        repo_id=REPO_ID, token=HF_TOKEN,
        commit_message=f'Upload LLM GGUF: {repo_name}',
    )
    print(f'  {repo_name} uploaded')

# ---- mmproj GGUF ----
mmproj_candidates = glob.glob('/content/mmproj_output/*.gguf') + ['/content/mmproj-f16.gguf']
for mp in mmproj_candidates:
    if os.path.exists(mp):
        size_mb = os.path.getsize(mp) / 1e6
        print(f'\nUploading mmproj: {os.path.basename(mp)} ({size_mb:.0f} MB)...')
        api.upload_file(
            path_or_fileobj=mp, path_in_repo='mmproj-f16.gguf',
            repo_id=REPO_ID, token=HF_TOKEN,
            commit_message='Upload mmproj GGUF (SigLIP-2 + MLP projector)',
        )
        print(f'  mmproj uploaded')
        break

print(f'\nDone! https://huggingface.co/{REPO_ID}')


## Step 13: Upload README with Offline Usage Guide


In [ ]:
readme = '''---
license: apache-2.0
tags:
  - gguf
  - vlm
  - multimodal
  - siglip2
  - qwen2
  - offline-inference
base_model: abhinav0231/Lily-1.5b-SFT-siglip2
---

# Lily VLM 1.5B - Complete GGUF (LLM + Vision + Projector)

Fully offline multimodal inference with two GGUF files.

## Architecture

| Component | Details | GGUF File |
|-----------|---------|-----------|
| Vision Encoder | SigLIP-2 (400M, 384px, patch14) | `mmproj-f16.gguf` |
| MLP Projector | Linear(1152->2048)->SiLU->Linear(2048->1536) | bundled in `mmproj-f16.gguf` |
| LLM Backbone | Qwen2-based 1.5B CausalLM | `Lily-VLM-1.5b-*.gguf` |

## Files

| File | Type | Size | Description |
|------|------|------|-------------|
| `mmproj-f16.gguf` | Vision+Proj | ~800 MB | SigLIP-2 encoder + MLP projector (F16) |
| `Lily-VLM-1.5b-F16.gguf` | LLM | ~3.1 GB | Full precision LLM backbone |
| `Lily-VLM-1.5b-Q4_K_M.gguf` | LLM | ~1.0 GB | Best speed/quality balance |
| `Lily-VLM-1.5b-Q5_K_M.gguf` | LLM | ~1.1 GB | Slightly better quality |
| `Lily-VLM-1.5b-Q8_0.gguf` | LLM | ~1.6 GB | Near-lossless |

## Offline Usage

### llama.cpp (recommended)
```bash
# Build llama.cpp
git clone https://github.com/ggerganov/llama.cpp && cd llama.cpp
cmake -B build && cmake --build build --config Release -j$(nproc)

# Run multimodal inference
./build/bin/llama-mtmd-cli \\
  -m Lily-VLM-1.5b-Q4_K_M.gguf \\
  --mmproj mmproj-f16.gguf \\
  --image photo.jpg \\
  -p "Describe this image in detail" \\
  -n 512 --temp 0.3
```

### Text-only (no image)
```bash
./build/bin/llama-cli \\
  -m Lily-VLM-1.5b-Q4_K_M.gguf \\
  -p "<|im_start|>system\\nYou are helpful<|im_end|>\\n<|im_start|>user\\nWhat is 2+2?<|im_end|>\\n<|im_start|>assistant\\n" \\
  -n 256 --temp 0.2
```

## Training Pipeline
1. Cold Start SFT Warmup -> GRPO RL -> Lily-1.5B
2. Offline Distillation -> Lily-1.5b-v0.3
3. Vision Data -> Projector Pretraining -> Vision SFT -> Lily-1.5b-SFT-siglip2
4. GGUF Conversion -> **This repo** (fully offline)
'''

readme_path = '/content/README.md'
with open(readme_path, 'w') as f:
    f.write(readme)

api.upload_file(
    path_or_fileobj=readme_path, path_in_repo='README.md',
    repo_id=REPO_ID, token=HF_TOKEN,
    commit_message='Add README with offline usage guide',
)
print('README uploaded')
print(f'\nAll done! Visit: https://huggingface.co/{REPO_ID}')


## Summary

You now have **two GGUF files** that together form a complete, fully offline VLM:

| File | Contains | Size |
|------|----------|------|
| `Lily-VLM-1.5b-Q4_K_M.gguf` | LLM backbone (Qwen2 1.5B, quantized) | ~1 GB |
| `mmproj-f16.gguf` | SigLIP-2 vision encoder + MLP projector | ~800 MB |

**Total: ~1.8 GB for a complete multimodal VLM running fully offline.**

### To run offline:
```bash
llama-mtmd-cli -m Lily-VLM-1.5b-Q4_K_M.gguf --mmproj mmproj-f16.gguf --image photo.jpg -p "Describe this"
```

No internet, no Python, no CUDA required - pure C++ inference.
